# Исследовательский разбор доменного сдвига на границе `O/B`

Этот ноутбук сравнивает два домена для горячей границы `O/B`:

- `train_time`: обучающий источник модели coarse-классификации, на котором текущий артефакт почти идеален;
- `downstream_pass`: рабочий поток после `quality_gate`, где класс `O` начинает схлопываться в `B`.


## Что делаем

1. Поднимаем типизированный набор данных для `train_time` и `downstream_pass`.
2. Сравниваем объемы и баланс истинных классов.
3. Проверяем предсказания текущей coarse-модели в обоих доменах.
4. Смотрим вероятности, физические признаки и долю пропусков.
5. Проверяем, есть ли доменный сдвиг, который объясняет проблему `O -> B`.


## Как читать результаты

- Если `train_time` и `downstream_pass` близки по физике и пропускам, а coarse-модель все равно теряет `O`, это модельный дефект.
- Если текущая coarse-модель почти идеальна на `train_time`, но ломается на `downstream_pass`, это признак доменного сдвига.
- Если физика downstream-объектов, размеченных как `O`, сильно ближе к `B`, чем к обучающему `O`, проблема сидит не в коде, а в источнике или смысле метки.


In [1]:
# Настройка: корень репозитория, `sys.path` и базовые параметры отображения.
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import seaborn as sns
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    # Ищем корень репозитория по каталогам `.git` и `src`.
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / '.git').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Не удалось определить корень репозитория из текущей рабочей директории.')


REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)


In [2]:
# Импортируем helpers после добавления `src` в `sys.path`.
from exohost.reporting.coarse_ob_domain_shift_review import (
    build_domain_class_balance_frame,
    build_domain_confusion_frame,
    build_domain_membership_summary_frame,
    build_domain_missingness_summary_frame,
    build_domain_physics_summary_frame,
    build_domain_predicted_class_summary_frame,
    build_domain_probability_summary_frame,
    build_domain_shift_auc_frame,
    load_coarse_ob_domain_shift_review_bundle_from_env,
)
from exohost.reporting.notebook_display import rename_frame_for_display
from exohost.reporting.notebook_labels import DOMAIN_NAME_LABELS, QUALITY_STATE_LABELS


In [3]:
# Загружаем артефакт coarse-модели и задаем словари для отображения.
COARSE_MODEL_RUN_DIR = (
    REPO_ROOT
    / 'artifacts'
    / 'models'
    / 'gaia_id_coarse_classification__hist_gradient_boosting__2026_03_28_215003_509969'
)
DOTENV_PATH = REPO_ROOT / '.env'

bundle = load_coarse_ob_domain_shift_review_bundle_from_env(
    coarse_model_run_dir=COARSE_MODEL_RUN_DIR,
    dotenv_path=str(DOTENV_PATH),
)


## Объемы и баланс классов

In [4]:
# Сравниваем объем и баланс классов в обучающем и рабочем доменах.
membership_df = build_domain_membership_summary_frame(bundle)
class_balance_df = build_domain_class_balance_frame(bundle)

display(
    rename_frame_for_display(
        membership_df,
        column_mapping={
            'quality_state': 'Состояние фильтра качества',
            'hot_teff_min_k': 'Минимальная температура горячего среза, K',
            'n_rows_train_boundary': 'Строк на границе обучающего домена',
            'n_rows_downstream_boundary': 'Строк на границе рабочего домена',
            'n_rows_train_scored': 'Строк с оценкой в обучающем домене',
            'n_rows_downstream_scored': 'Строк с оценкой в рабочем домене',
        },
        value_mapping={'quality_state': QUALITY_STATE_LABELS},
    )
)


,Состояние фильтра качества,"Минимальная температура горячего среза, K",Строк на границе обучающего домена,Строк на границе рабочего домена,Строк с оценкой в обучающем домене,Строк с оценкой в рабочем домене
0,Допуск,10000.0,6000,7137,6000,7137


In [5]:
display(
    rename_frame_for_display(
        class_balance_df,
        column_mapping={
            'domain_name': 'Домен',
            'true_spectral_class': 'Истинный спектральный класс',
            'n_rows': 'Число строк',
            'share_within_group': 'Доля внутри домена',
        },
        value_mapping={'domain_name': DOMAIN_NAME_LABELS},
    )
)


,Домен,Истинный спектральный класс,Число строк,Доля внутри домена
0,Рабочий домен,B,7112,0.996497
1,Рабочий домен,O,25,0.003503
2,Обучающий домен,B,3000,0.500000
3,Обучающий домен,O,3000,0.500000


## Предсказания текущей coarse-модели

In [6]:
predicted_df = build_domain_predicted_class_summary_frame(bundle)
confusion_df = build_domain_confusion_frame(bundle)

display(
    rename_frame_for_display(
        predicted_df,
        column_mapping={
            'domain_name': 'Домен',
            'coarse_predicted_label': 'Предсказанный класс coarse-модели',
            'n_rows': 'Число строк',
            'share_within_group': 'Доля внутри домена',
        },
        value_mapping={'domain_name': DOMAIN_NAME_LABELS},
    )
)


,Домен,Предсказанный класс coarse-модели,Число строк,Доля внутри домена
0,Рабочий домен,B,7116,0.997058
1,Рабочий домен,A,21,0.002942
2,Обучающий домен,O,3001,0.500167
3,Обучающий домен,B,2995,0.499167
4,Обучающий домен,A,4,0.000667


In [7]:
display(
    rename_frame_for_display(
        confusion_df,
        column_mapping={
            'domain_name': 'Домен',
            'true_spectral_class': 'Истинный спектральный класс',
            'coarse_predicted_label': 'Предсказанный класс coarse-модели',
            'n_rows': 'Число строк',
            'share_within_true_class': 'Доля внутри истинного класса',
        },
        value_mapping={'domain_name': DOMAIN_NAME_LABELS},
    )
)


,Домен,Истинный спектральный класс,Предсказанный класс coarse-модели,Число строк,Доля внутри истинного класса
0,Рабочий домен,B,B,7091,0.997047
1,Рабочий домен,B,A,21,0.002953
2,Рабочий домен,O,B,25,1.000000
3,Обучающий домен,B,B,2995,0.998333
4,Обучающий домен,B,A,4,0.001333
5,Обучающий домен,B,O,1,0.000333
6,Обучающий домен,O,O,3000,1.000000


## Вероятности `P(O)` и `P(B)`

In [8]:
probability_df = build_domain_probability_summary_frame(bundle)
display(
    rename_frame_for_display(
        probability_df,
        column_mapping={
            'domain_name': 'Домен',
            'true_spectral_class': 'Истинный спектральный класс',
            'n_rows': 'Число строк',
            'median_probability__O': 'Медианная вероятность P(O)',
            'median_probability__B': 'Медианная вероятность P(B)',
            'mean_probability__O': 'Средняя вероятность P(O)',
            'mean_probability__B': 'Средняя вероятность P(B)',
        },
        value_mapping={'domain_name': DOMAIN_NAME_LABELS},
    )
)


,Домен,Истинный спектральный класс,Число строк,Медианная вероятность P(O),Медианная вероятность P(B),Средняя вероятность P(O),Средняя вероятность P(B)
0,Рабочий домен,B,7112,0.000017,0.999842,0.000031,0.997244
1,Рабочий домен,O,25,0.000017,0.999842,0.000086,0.999751
2,Обучающий домен,B,3000,0.000017,0.999842,0.000378,0.997463
3,Обучающий домен,O,3000,0.999845,0.000018,0.999668,0.000189


## Физика по доменам и истинным классам

In [9]:
physics_df = build_domain_physics_summary_frame(bundle)
display(
    rename_frame_for_display(
        physics_df,
        column_mapping={
            'domain_name': 'Домен',
            'true_spectral_class': 'Истинный спектральный класс',
            'n_rows': 'Число строк',
            'median_teff_gspphot': 'Медианная температура `GSP-Phot`',
            'median_logg_gspphot': 'Медианный `logg` `GSP-Phot`',
            'median_mh_gspphot': 'Медианная металличность `GSP-Phot`',
            'median_bp_rp': 'Медианный цвет BP-RP',
            'median_parallax': 'Медианный параллакс',
            'median_parallax_over_error': 'Медианный SNR параллакса',
            'median_ruwe': 'Медианный RUWE',
            'median_radius_feature': 'Медианный рабочий радиус',
            'median_radius_flame': 'Медианный радиус FLAME',
        },
        value_mapping={'domain_name': DOMAIN_NAME_LABELS},
    )
)


,Домен,Истинный спектральный класс,Число строк,Медианная температура `GSP-Phot`,Медианный `logg` `GSP-Phot`,Медианная металличность `GSP-Phot`,Медианный цвет BP-RP,Медианный параллакс,Медианный SNR параллакса,Медианный RUWE,Медианный рабочий радиус,Медианный радиус FLAME
0,Рабочий домен,B,7112,12311.6575,3.78990,-0.50570,0.198108,0.690217,33.019384,0.985116,3.530580,3.530580
1,Рабочий домен,O,25,15080.1200,3.70180,-0.21940,0.450726,0.277424,11.610753,1.041786,4.070290,4.070290
2,Обучающий домен,B,3000,12355.9140,3.77930,-0.46645,1.258621,0.256354,7.635526,1.010951,3.890203,3.791376
3,Обучающий домен,O,3000,34604.6170,3.89625,0.01860,3.385037,0.138732,4.884386,1.024742,8.796550,NaN


## Пропуски по признакам

In [10]:
missingness_df = build_domain_missingness_summary_frame(bundle)

missingness_gap_df = (
    missingness_df
    .pivot_table(
        index=['true_spectral_class', 'feature_name'],
        columns='domain_name',
        values='missing_share',
        aggfunc='first',
    )
    .reset_index()
    .rename_axis(columns=None)
)
for column_name in ['train_time', 'downstream_pass']:
    if column_name not in missingness_gap_df.columns:
        missingness_gap_df[column_name] = 0.0
missingness_gap_df['missing_share_gap'] = (
    missingness_gap_df['downstream_pass'] - missingness_gap_df['train_time']
)
missingness_gap_df['missing_share_gap_abs'] = missingness_gap_df['missing_share_gap'].abs()

missingness_gap_view_df = missingness_gap_df.sort_values(
    ['missing_share_gap_abs', 'true_spectral_class', 'feature_name'],
    ascending=[False, True, True],
    ignore_index=True,
)

display(
    missingness_gap_view_df.rename(
        columns={
            'true_spectral_class': 'Истинный спектральный класс',
            'feature_name': 'Признак',
            'train_time': 'Доля пропусков в обучающем домене',
            'downstream_pass': 'Доля пропусков в рабочем домене',
            'missing_share_gap': 'Разница долей пропусков',
            'missing_share_gap_abs': 'Модуль разницы долей пропусков',
        }
    )
)


,Истинный спектральный класс,Признак,Доля пропусков в рабочем домене,Доля пропусков в обучающем домене,Разница долей пропусков,Модуль разницы долей пропусков
0,B,bp_rp,0.0,0.0,0.0,0.0
1,B,logg_gspphot,0.0,0.0,0.0,0.0
2,B,mh_gspphot,0.0,0.0,0.0,0.0
3,B,parallax,0.0,0.0,0.0,0.0
4,B,parallax_over_error,0.0,0.0,0.0,0.0
5,B,radius_feature,0.0,0.0,0.0,0.0
6,B,ruwe,0.0,0.0,0.0,0.0
7,B,teff_gspphot,0.0,0.0,0.0,0.0
8,O,bp_rp,0.0,0.0,0.0,0.0
9,O,logg_gspphot,0.0,0.0,0.0,0.0


## Насколько различаются обучающий и рабочий домены

In [11]:
shift_auc_df = build_domain_shift_auc_frame(bundle)

display(
    rename_frame_for_display(
        shift_auc_df.sort_values(
            ['separability_auc', 'true_spectral_class', 'feature_name'],
            ascending=[False, True, True],
            ignore_index=True,
        ),
        column_mapping={
            'true_spectral_class': 'Истинный спектральный класс',
            'feature_name': 'Признак',
            'n_rows_used': 'Число строк в расчете',
            'auc_ovr_downstream': 'AUC для рабочего против обучающего домена',
            'separability_auc': 'Сила разделения доменов',
            'higher_value_domain': 'Домен с более высокими значениями',
            'median_train_time': 'Медиана в обучающем домене',
            'median_downstream_pass': 'Медиана в рабочем домене',
        },
        value_mapping={'higher_value_domain': DOMAIN_NAME_LABELS},
    )
)


,Истинный спектральный класс,Признак,Число строк в расчете,AUC для рабочего против обучающего домена,Сила разделения доменов,Домен с более высокими значениями,Медиана в обучающем домене,Медиана в рабочем домене
0,O,teff_gspphot,3025,0.000000,1.000000,Обучающий домен,34604.617000,15080.120000
1,O,bp_rp,3025,0.010520,0.989480,Обучающий домен,3.385037,0.450726
2,B,parallax_over_error,10112,0.899968,0.899968,Рабочий домен,7.635526,33.019384
3,O,radius_feature,3025,0.101427,0.898573,Обучающий домен,8.796550,4.070290
4,B,bp_rp,10112,0.107991,0.892009,Обучающий домен,1.258621,0.198108
5,B,parallax,10112,0.869086,0.869086,Рабочий домен,0.256354,0.690217
6,O,parallax_over_error,3025,0.828813,0.828813,Рабочий домен,4.884386,11.610753
7,O,parallax,3025,0.812800,0.812800,Рабочий домен,0.138732,0.277424
8,O,mh_gspphot,3025,0.354013,0.645987,Обучающий домен,0.018600,-0.219400
9,O,logg_gspphot,3025,0.391260,0.608740,Обучающий домен,3.896250,3.701800


## Краткий вывод

Текущий обзор подтверждает сильный доменный сдвиг между обучающим и рабочим горячими доменами `O/B`.

Ключевые выводы:

- текущая coarse-модель почти идеальна на собственном обучающем домене;
- на рабочем домене downstream-объекты, размеченные как `O`, полностью уходят в `B`;
- этот сдвиг нельзя объяснить только пропусками;
- сильнее всего различаются температура, цвет, радиус и точность параллакса.

Итог: проблема выглядит как доменный сдвиг и расхождение в источнике меток, а не как простой дефект модели.